*This is the notebook for the inference phase of this example*

## Feed Forward Network Demonstration (For MNIST)
**Goal:** Recognise the digit drawn on the image, shooting for 90%+ accuracy on Kaggle.

**Strategy:**
1. We load our training result from a `.cache` file.
2. We read from the `test.csv` file and return our answer to `result.csv` file
3. Then we send to Kaggle to evaluate the performance.

This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [1]:
import pandas as pd
import time
import lktorch as lk
import os

CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")
DATA_PATH = os.path.join(CURRENT_DIR, "training_data", "test.csv")
RESULT_PATH = os.path.join(CURRENT_DIR, "result", "result.csv")

### 2. Generate and preprocess

In [2]:

input = pd.read_csv(DATA_PATH)
row_count, column_count = input.shape

input_dataset = []
label = []

for i in range(row_count):
    input_row = input.loc[i].to_numpy()
    input_dataset.append(input_row)


def preprocess_row(current_row):
    result_row = [0] * 784
    moment_x = 0
    moment_y = 0
    sum = 0
    for i in range(0, 784):
        x = i // 28
        y = i % 28
        moment_x += x * current_row[i]
        moment_y += y * current_row[i]
        sum += current_row[i]
    centroid_x = moment_x // sum
    centroid_y = moment_y // sum

    centroid_x = 14 - centroid_x
    centroid_y = 14 - centroid_y

    for i in range(0, 784):
        x = i // 28
        y = i % 28

        x += centroid_x
        y += centroid_y

        if (x >= 0 and x < 28 and y >= 0 and y < 28):
            result_row[x * 28 + y] = current_row[i]

    ma = 0
    for i in result_row:
        ma = max(ma, i)
    for i in range(0, 784):
        result_row[i] = result_row[i] / ma

    return result_row

for i in range(len(input_dataset)):
    input_dataset[i] = preprocess_row(input_dataset[i])

print("Done fetching data!")

Done fetching data!


### 3. Inference

In [ ]:

def get_label(tensor):
    label = []
    sz = tensor.dimension()
    for i in range(sz[0]):
        cur = -1
        ma = -1
        for j in range(0, 10):
            current_w = tensor.accessA([i, j])
            if (current_w > ma):
                ma = current_w
                cur = j
        label.append(cur)
    return label

lk.deactivate_backprop()


model = lk.Sequential([
    lk.Reshape_Layer([28, 28]),
    lk.Unfold_Layer(4, 4),
    lk.Conv2D([4, 4], [3]),
    lk.Reshape_Layer([147]),
    lk.reLU_Layer(),
    
    lk.LinearLayer(147, 32),
    lk.reLU_Layer(),
    lk.LinearLayer(32, 16),
    lk.reLU_Layer(),
    lk.LinearLayer(16, 10),
    lk.Sigmoid_Layer(),
    lk.SoftMax_Layer()
])


model.load_state_dict(lk.ReadFile(WEIGHT_PATH))
input_tensor = lk.Tensor(input_dataset)
tenso = model(input_tensor)
label = get_label(tenso)

output_d = {"ImageId": [i for i in range(1, row_count + 1)], "Label": label}
data_frame = pd.DataFrame(data = output_d)
print(data_frame)
data_frame.to_csv(RESULT_PATH, index=False)